# 03 · Evaluation on the held-out test split

Evaluates Random Forest, XGBoost and the **ensemble (fusion rule)** used by the ASTRA API:
accuracy, macro/weighted precision-recall-F1, per-class report and confusion matrix, an ablation of the
fusion rule, and a ready-to-paste LaTeX table. All numbers are computed here; nothing is hard-coded.

**Needs:** outputs of notebooks 01 and 02.

In [ ]:
import os, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

WORK_DIR = Path(os.environ.get("ASTRA_WORK_DIR", "astra_run"))   # shared by all four notebooks
ART_DIR = WORK_DIR / "artifacts"
SEED = 42
CLASSES = ["benign", "bruteforce", "ddos", "dos", "portscan"]   # alphabetical = LabelEncoder order
ART_DIR.mkdir(parents=True, exist_ok=True)

def fuse(rf_labels, xgb_labels):
    """Vectorised version of app/routers/predict.py::make_verdict.
    agree -> that label | one says benign -> the other (threat) | two different threats -> RF."""
    rf_labels, xgb_labels = np.asarray(rf_labels), np.asarray(xgb_labels)
    out = rf_labels.copy()
    use_xgb = (rf_labels == "benign") & (xgb_labels != "benign")
    out[use_xgb] = xgb_labels[use_xgb]
    return out


def make_verdict(rf_label, xgb_label):          # reference implementation from the API
    if rf_label == xgb_label: return rf_label
    if rf_label == "benign": return xgb_label
    if xgb_label == "benign": return rf_label
    return rf_label

from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             precision_recall_fscore_support)
S = joblib.load(WORK_DIR / "splits.joblib")
le = joblib.load(ART_DIR / "label_encoder.pkl")
rf = joblib.load(ART_DIR / "rf_multiclass.pkl")
xgb = joblib.load(ART_DIR / "xgb_multiclass.pkl")
data_meta = json.loads((WORK_DIR / "data_meta.json").read_text())
X_te, y_te = S["X_test"], S["y_test"]
y_true = le.inverse_transform(y_te)
print("test rows:", len(y_true), "| smoke run:", data_meta["split"]["smoke"])

## Predictions

In [ ]:
rf_lab = le.inverse_transform(rf.predict(X_te))
xgb_lab = le.inverse_transform(xgb.predict(X_te))
ens_lab = fuse(rf_lab, xgb_lab)

# the vectorised rule must equal the API's reference implementation
chk = np.random.RandomState(0).choice(len(ens_lab), min(20000, len(ens_lab)), replace=False)
assert all(ens_lab[i] == make_verdict(rf_lab[i], xgb_lab[i]) for i in chk)
print("fusion rule check passed")

## Overall metrics

In [ ]:
def summarize(name, y_t, y_p):
    p, r, f, s = precision_recall_fscore_support(y_t, y_p, labels=CLASSES, zero_division=0)
    pw, rw, fw, _ = precision_recall_fscore_support(y_t, y_p, average="weighted", zero_division=0)
    return {"model": name, "accuracy": accuracy_score(y_t, y_p),
            "macro_precision": p.mean(), "macro_recall": r.mean(), "macro_f1": f.mean(),
            "weighted_precision": pw, "weighted_recall": rw, "weighted_f1": fw,
            "per_class": {c: {"precision": p[i], "recall": r[i], "f1": f[i], "support": int(s[i])}
                          for i, c in enumerate(CLASSES)}}

results = [summarize("Random Forest", y_true, rf_lab), summarize("XGBoost", y_true, xgb_lab),
           summarize("Ensemble (fusion)", y_true, ens_lab)]
overall = pd.DataFrame([{k: v for k, v in r.items() if k != "per_class"} for r in results]).set_index("model")
display((overall * 100).round(2))

## Per-class report

In [ ]:
for name, lab in [("Random Forest", rf_lab), ("XGBoost", xgb_lab), ("Ensemble (fusion)", ens_lab)]:
    print("=" * 20, name, "=" * 20)
    print(classification_report(y_true, lab, labels=CLASSES, digits=4, zero_division=0))

## Confusion matrix (ensemble)

In [ ]:
cm = confusion_matrix(y_true, ens_lab, labels=CLASSES)
cmn = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(5), CLASSES, rotation=35, ha="right"); ax.set_yticks(range(5), CLASSES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Ensemble confusion matrix (row-normalised)")
for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", fontsize=7, color="white" if cmn[i, j] > .5 else "black")
fig.colorbar(im, fraction=0.046); plt.tight_layout()
fig.savefig(WORK_DIR / "confusion_matrix_ensemble.png", dpi=200); plt.show()
pd.DataFrame(cm, index=[f"true_{c}" for c in CLASSES], columns=[f"pred_{c}" for c in CLASSES]) \
    .to_csv(WORK_DIR / "confusion_matrix_ensemble.csv")

## Does the fusion rule help? (ablation)

In [ ]:
rf_ok, xgb_ok, ens_ok = rf_lab == y_true, xgb_lab == y_true, ens_lab == y_true
n = len(y_true)
abl = pd.DataFrame({
    "rows": [n, int((rf_lab != xgb_lab).sum()), int((ens_ok & ~rf_ok).sum()), int((ens_ok & ~xgb_ok).sum()),
             int((~ens_ok & rf_ok).sum()), int((~ens_ok & xgb_ok).sum())]},
    index=["test rows", "RF and XGBoost disagree", "ensemble right, RF wrong", "ensemble right, XGB wrong",
           "ensemble wrong, RF right", "ensemble wrong, XGB right"])
display(abl)
best_single = max(results[0]["accuracy"], results[1]["accuracy"])
print(f"ensemble accuracy - best single model = {(results[2]['accuracy'] - best_single) * 100:+.4f} percentage points")

## LaTeX table for the paper (generated from the numbers above)

In [ ]:
def pct(x): return f"{100 * x:.2f}"
rows = []
for c in CLASSES:
    pc = results[2]["per_class"][c]
    rows.append(f"{c} & {pct(pc['precision'])} & {pct(pc['recall'])} & {pct(pc['f1'])} & {pc['support']:,} \\\\")
tex = r"""\begin{table}[t]
\centering
\caption{Per-class results of the ASTRA ensemble on the held-out test split.}
\label{tab:perclass}
\begin{tabular}{lrrrr}
\hline
Class & Precision (\%) & Recall (\%) & F1 (\%) & Support \\
\hline
""" + "\n".join(rows) + r"""
\hline
\end{tabular}
\end{table}
"""
rows2 = [f"{r['model']} & {pct(r['accuracy'])} & {pct(r['macro_precision'])} & {pct(r['macro_recall'])} & {pct(r['macro_f1'])} \\\\"
         for r in results]
tex2 = r"""\begin{table}[t]
\centering
\caption{Overall test performance (macro-averaged over the five classes).}
\label{tab:overall}
\begin{tabular}{lrrrr}
\hline
Model & Acc. (\%) & Prec. (\%) & Rec. (\%) & F1 (\%) \\
\hline
""" + "\n".join(rows2) + r"""
\hline
\end{tabular}
\end{table}
"""
(WORK_DIR / "tables.tex").write_text(tex2 + "\n" + tex, encoding="utf-8")
print(tex2); print(tex)

## Save everything

In [ ]:
def clean(o):
    if isinstance(o, dict): return {k: clean(v) for k, v in o.items()}
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.integer,)): return int(o)
    return o

out = {"data": data_meta, "results": clean(results), "confusion_matrix_ensemble": {"labels": CLASSES, "matrix": cm.tolist()}}
(WORK_DIR / "metrics_eval.json").write_text(json.dumps(out, indent=2), encoding="utf-8")
print("saved metrics_eval.json, confusion_matrix_ensemble.csv/.png, tables.tex in", WORK_DIR.resolve())
if data_meta["split"]["smoke"]: print("SMOKE run - do not use these results in the paper.")